In [2]:
import numpy as np
import pandas as pd
import random
from collections import deque
import tensorflow as tf
import networkx as nx
import os
import matplotlib.pyplot as plt
import csv

# 현재 작업 디렉토리 가져오기
current_dir = os.getcwd()

# 폴더 경로 생성
data_dir = os.path.join(current_dir, 'data')
result_dir = os.path.join(current_dir, 'results')

# 폴더 경로 내 파일 경로 생성
passenger_data_file = os.path.join(data_dir, 'passengers.csv')
vehicle_positions_file = os.path.join(data_dir, 'vehicle_positions.csv')

# 네트워크 및 Travel Time Matrix 생성
class SiouxFallsNetwork:
    def __init__(self, net_file, flow_file, node_coord_file, node_xy_file):
        self.sioux_falls_df, self.node_coord, self.node_xy = self.load_data(net_file, flow_file, node_coord_file, node_xy_file)
        self.graph = self.create_graph()
        self.travel_time = self.initialize_travel_time()
        nx.set_edge_attributes(self.graph, self.travel_time, "weight")  # 초기화 후 고정

    def load_data(self, net_file, flow_file, node_coord_file, node_xy_file):
        net = pd.read_csv(net_file, skiprows=8, sep='\t').drop(['~', ';'], axis=1, errors='ignore')
        net['edge'] = net.index + 1
        flow = pd.read_csv(flow_file, sep='\t').drop(['From ', 'To '], axis=1, errors='ignore')
        flow.rename(columns={"Volume ": "flow", "Cost ": "cost"}, inplace=True)
        node_coord = pd.read_csv(node_coord_file, sep='\t').drop([';'], axis=1, errors='ignore')
        node_xy = pd.read_csv(node_xy_file, sep='\t')

        sioux_falls_df = pd.concat([net, flow], axis=1)
        return sioux_falls_df, node_coord, node_xy

    def create_graph(self):
        G = nx.from_pandas_edgelist(self.sioux_falls_df, 'init_node', 'term_node',
                                    ['capacity', 'length', 'free_flow_time', 'b', 'power', 'speed', 'toll', 'link_type', 'edge', 'flow', 'cost'],
                                    create_using=nx.MultiDiGraph())

        # Coordinate position (using pos_xy for better visualization)
        pos_xy = dict([(i, (a, b)) for i, a, b in zip(self.node_xy.Node, self.node_xy.X, self.node_xy.Y)])
        nx.set_node_attributes(G, pos_xy, 'pos')

        return G

    def initialize_travel_time(self):
        """
        네트워크 엣지에 대해 travel_time을 랜덤 값(1~3)으로 초기화합니다.
        """
        travel_time = {}
        for u, v, k, data in self.graph.edges(data=True, keys=True):
            # 항상 랜덤 값을 사용
            random_time = np.random.randint(1, 4)  # 1~3 사이의 랜덤 값
            travel_time[(u, v, k)] = random_time
            #print(f"Edge ({u}, {v}, {k}): travel_time={random_time}")  # 디버깅 출력

        nx.set_edge_attributes(self.graph, travel_time, "weight")
        self.travel_time = travel_time  # 속성으로 저장
        return travel_time

    def get_shortest_path(self, start, end):
        try:
            path = nx.shortest_path(self.graph, source=start, target=end, weight="weight")
            return path
        except nx.NetworkXNoPath:
            print(f"No path exists between Node {start} and Node {end}.")
            return []

    def get_travel_time(self, path):
        if len(path) < 2:
            return float('inf')
        total_time = 0
        for i in range(len(path) - 1):
            edges = self.graph.get_edge_data(path[i], path[i + 1])
            if edges:
                total_time += min([edge_data.get('weight', float('inf')) for key, edge_data in edges.items()])
            else:
                print(f"No edge exists between Node {path[i]} and Node {path[i + 1]}.")
                return float('inf')
        return total_time

    def save_travel_time(self, output_file):
        """
        Travel Time 데이터를 CSV 파일로 저장합니다.
        """
        with open(output_file, 'w', newline='') as csvfile:
            writer = csv.writer(csvfile)
            writer.writerow(['From', 'To', 'Key', 'TravelTime'])
            for (u, v, k), travel_time in self.travel_time.items():
                writer.writerow([u, v, k, travel_time])
        print(f"Travel time data saved to {output_file}")

    def generate_od_matrix(self, output_file):
        """
        OD 최소 이동 시간 행렬을 CSV 파일로 저장합니다.
        """
        nodes = list(self.graph.nodes)
        num_nodes = len(nodes)
        od_matrix = np.full((num_nodes, num_nodes), np.inf)  # 초기값은 무한대로 설정

        for i, origin in enumerate(nodes):
            for j, destination in enumerate(nodes):
                if origin == destination:
                    od_matrix[i, j] = 0  # 자기 자신으로의 이동 시간은 0
                else:
                    try:
                        # 최단 경로의 가중치 합 계산
                        travel_time = nx.shortest_path_length(self.graph, source=origin, target=destination, weight="weight")
                        od_matrix[i, j] = travel_time
                    except nx.NetworkXNoPath:
                        pass  # 경로가 없는 경우 무한대 유지

        # 행렬을 CSV 파일로 저장
        pd.DataFrame(od_matrix, index=nodes, columns=nodes).to_csv(output_file, index_label='Origin')
        print(f"OD matrix saved to {output_file}")

class Passenger:
    def __init__(self, id, start, end, request_time, network):
        self.id = id
        self.start = start
        self.end = end
        self.request_time = request_time
        self.pickup_time = None
        self.dropoff_time = None
        
        # [추가] 승객의 최단 경로 이동시간 -> 우회 비용 계산에 사용
        self.network = network
        self.direct_route_time = 0
        if network is not None:
            path = network.get_shortest_path(start, end)
            if path:
                self.direct_route_time = network.get_travel_time(path)

class Vehicle:
    def __init__(self, id, capacity, current_location, network, env):
        self.id = id
        self.capacity = capacity
        self.current_location = current_location
        self.network = network
        self.env = env  # RideSharingEnvironment 객체를 참조
        self.passengers = []
        self.remaining_travel_time = 0
        self.current_path = []
        self.rebalance_target = None
        self.rebalance_count = 0  # Rebalancing 횟수 추적 속성 추가
        self.idle_time = 0  # 차량이 유휴 상태로 있는 시간
        self.dropped_passengers = 0  # 하차한 승객 수를 추적하는 변수 추가
        self.status = 'idle' # 250321 추가
        
    def add_passenger(self, passenger):
        # 수정: capacity 조건 "<" 로 변경
        if len(self.passengers) < self.capacity:
            self.passengers.append(passenger)
            self.idle_time = 0  # 승객을 태우면 유휴 시간 초기화
            return True
        return False

    def remove_passenger(self, passenger):
        if passenger in self.passengers:
            self.passengers.remove(passenger)

    def move_to_next_location(self):
        dropped = [] # 하차한 승객을 담을 리스트
        # 차량이 이동 중인 경우, 남은 이동 시간을 감소
        if self.remaining_travel_time > 0:
            self.remaining_travel_time -= 1

        # 이동이 완료된 경우 다음 경로로 이동
        if self.remaining_travel_time == 0 and self.current_path:
            self.current_location = self.current_path.pop(0)

            # 다음 목적지가 있으면 이동 시간 설정
            if self.current_path:
                next_node = self.current_path[0]
                self.remaining_travel_time = self.network.get_travel_time([self.current_location, next_node])
            else:
                # 경로가 더 이상 없을 때 추가 이동 불가 (재배치 또는 대기)
                self.remaining_travel_time = 0

        # 차량이 유휴 상태일 경우 idle_time 증가
        if not self.current_path and self.remaining_travel_time == 0:
            self.idle_time += 1

        # 중간 하차 지점에서 승객을 내리기
        for passenger in self.passengers[:]:  # 승객 리스트 순회
            if passenger.end == self.current_location:  # 하차 지점에 도달
                self.remove_passenger(passenger)
                passenger.dropoff_time = self.env.time
                self.dropped_passengers += 1  # 하차 승객 수 증가
                self.env.dropped_passengers += 1  # 환경의 하차 승객 수도 증가
                dropped.append(passenger)  # 하차 승객 리스트에 추가
                #print(f"Passenger {passenger.id} dropped off at {self.current_location} by Vehicle {self.id}")
        return dropped

class RideSharingEnvironment:
    def __init__(self, network, num_vehicles, capacity, passenger_data, vehicle_positions):
        self.network = network
        self.vehicles = self.initialize_vehicles(num_vehicles, capacity, vehicle_positions)
        self.passenger_data = passenger_data
        self.passengers = []  # 현재 활성 승객
        self.time = 0
        self.canceled_passengers = 0  # 취소된 승객 수 누적 (카운터)
        self.rebalancing_count = 0
        self.total_passenger_id = 0
        self.matched_passengers = 0  # 누적 매칭 카운터 (아래 집합으로 관리)
        self.dropped_passengers = 0
        self.last_dropped = []  # 마지막 하차 승객 리스트
        
        # 고수요 노드와 대기 승객 정보 초기화
        self.high_demand_nodes = []
        self.waiting_passenger_count = np.zeros(len(self.network.graph.nodes))
        
        # Reward 계산을 위한 여러 가중치들
        # self.alpha_qos               # qos 비용 가중치
        # self.alpha_rebalance =       # 재배치 비용 가중치
        # self.alpha_occupant =        # 승객 운행 비용 가중치
        # self.alpha_emission =        # 배출 가스 비용 가중치 
        self.base_fare = 1          # 기본 요금
        self.addttional_charge = 167   # 추가 요금
        # self.VOT =                   # 시간 가치
        self.VOT = 1.0

        # 매칭, 대기 승객 관리를 위한 집합 (유니크 카운팅)
        self.matched_ids = set()
        self.canceled_ids = set()
        self.dropped_ids = set()  # 추가: dropoff된 승객 ID 집합

        # 승객 데이터의 최대 요청 시간 계산 (동적 시뮬레이션 종료 조건에 사용)
        self.max_request_time = self.passenger_data['Request_time'].max()

        # [추가] 각 항목별 reward를 저장한 변수
        self.matched_reward = 0.0
        self.dropoff_reward = 0.0
        self.qos_reward = 0.0
        self.rebalancing_reward = 0.0
        self.emission_reward = 0.0

    def initialize_vehicles(self, num_vehicles, capacity, vehicle_positions):
        vehicles = []
        for idx, pos in enumerate(vehicle_positions):
            vehicles.append(Vehicle(idx, capacity, pos, self.network, self))
        return vehicles

    def initialize_passengers(self, passenger_data):
        passengers = []
        for _, row in passenger_data.iterrows():
            passenger = Passenger(
                id=row['User_ID'],
                start=row['Start_node'],
                end=row['End_node'],
                request_time=row['Request_time'],
                network=self.network  # 네트워크 전달
            )
            passengers.append(passenger)
        return passengers

    def get_state(self):
        # 상태 배열 초기화
        state = np.zeros(len(self.network.graph.nodes) * 3 + len(self.vehicles))
        
        # 차량 위치 정보
        for vehicle in self.vehicles:
            adjusted_location = int(vehicle.current_location) - 1  # 1부터 시작하는 값을 0부터 시작으로 변환
            if 0 <= adjusted_location < len(self.network.graph.nodes):
                state[adjusted_location] += 1
    
        # 대기 승객 정보 포함 (각 노드에서 대기 중인 승객 수)
        for i, count in enumerate(self.waiting_passenger_count):
            if 0 <= i < len(self.network.graph.nodes):
                state[len(self.network.graph.nodes) + i] = count
        
        # 고수요 노드 정보 포함
        for node in self.high_demand_nodes:
            adjusted_node = int(node) - 1  # 고수요 노드도 1부터 시작한다고 가정
            if 0 <= adjusted_node < len(self.network.graph.nodes):
                state[len(self.network.graph.nodes) * 2 + adjusted_node] += 5  # 고수요 노드에는 5 추가
    
        # 각 차량에 탑승한 승객 수
        for i, vehicle in enumerate(self.vehicles):
            state[len(self.network.graph.nodes) * 3 + i] = len(vehicle.passengers)
    
        return state

    # 새로운 step 함수
    def step(self, actions):
        old_rebalancing_count = self.rebalancing_count
        # HS: 해당 스텝에서 새로 매칭된 승객 수 카운트
        step_matched_count = 0
    
        # 1) 현재 시간에 맞는 승객 생성 (중복 생성 방지)
        # 단, env.time이 max_request_time를 넘으면 더 이상 생성하지 않음
        if self.time <= self.max_request_time:
            self.generate_passengers_for_current_time()
        
        # 2) 승객 요청 취소 로직: 취소 조건 만족 시 canceled_ids에 추가하고 제거
        passengers_to_remove = set()
        cancel_time_limit = 10
        for passenger in self.passengers[:]:
            if self.time - passenger.request_time > cancel_time_limit and passenger.pickup_time is None:
                if passenger.id not in self.canceled_ids:
                    self.canceled_ids.add(passenger.id)
                self.canceled_passengers = len(self.canceled_ids)
                passengers_to_remove.add(passenger)
                self.passengers.remove(passenger)
    
        # 차량별 하차 승객을 모으기 위한 리스트
        all_dropped = []
    
        # 3) 차량 액션(이동/재배치) 처리
        for vehicle, action in zip(self.vehicles, actions):
            # 이동/재배치 처리: 재배치인 경우에만 재배치 카운트 증가
            if vehicle.remaining_travel_time == 0 and not vehicle.current_path:
                if vehicle.rebalance_target is not None:
                    # 재배치인 경우: rebalancing_target 지정됨
                    path = self.network.get_shortest_path(vehicle.current_location, vehicle.rebalance_target)
                    vehicle.rebalance_target = None
                    if path:
                        vehicle.current_path = path[1:]
                        if vehicle.current_path:
                            next_node = vehicle.current_path[0]
                            vehicle.remaining_travel_time = self.network.get_travel_time([vehicle.current_location, next_node])
                            vehicle.rebalance_count += 1
                            self.rebalancing_count += 1
                else:
                    # 일반 이동: 재배치 카운트 증가 없음
                    path = self.network.get_shortest_path(vehicle.current_location, action)
                    if path:
                        vehicle.current_path = path[1:]
                        if vehicle.current_path:
                            next_node = vehicle.current_path[0]
                            vehicle.remaining_travel_time = self.network.get_travel_time([vehicle.current_location, next_node])
                            
            # 차량 이동 및 하차 처리: 하차한 승객은 리스트로 반환됨
            dropped = vehicle.move_to_next_location()
            all_dropped.extend(dropped)
            
            if dropped:  # 250321 추가: 하차가 발생하면 상태 변경
                vehicle.status = 'dropoff'
            elif vehicle.passengers:
                vehicle.status = 'moving'
            elif not vehicle.passengers and not vehicle.current_path and vehicle.remaining_travel_time == 0:
                vehicle.status = 'idle'
            if vehicle.status == 'idle' and vehicle.rebalance_target is not None:
                vehicle.status = 'rebalance'
    
            # 승객 탑승 처리: 대기 승객(self.passengers) 중 해당 위치의 승객을 차량에 태우고,
            # 중복 매칭 방지를 위해 탑승한 승객은 즉시 환경 리스트에서 제거
            for passenger in self.passengers[:]:
                if passenger.start == vehicle.current_location and passenger.pickup_time is None:
                    if vehicle.add_passenger(passenger):
                        passenger.pickup_time = self.time
                        if passenger.id not in self.matched_ids:
                            self.matched_ids.add(passenger.id)
                        step_matched_count += 1 
                        self.passengers.remove(passenger)
                        vehicle.status = 'pickup'  # 250321 추가
            
        # 4) 비용 계산
        total_cost = 0.0

        # [추가] 각 항목별 reward 저장

        # 1. 매칭 승객 보상 (매칭된 수/ 누적 요청수)
        cumulative_requests = len(self.matched_ids) + len(self.canceled_ids) + len(self.passengers)
        if cumulative_requests == 0:
            matching_ratio = 0
        else:
            matching_ratio = len(self.matched_ids) / cumulative_requests
        
        matched_reward = step_matched_count * self.base_fare * matching_ratio
        
        # 2. 하차 승객 보상 (현재 내려준 승객수 * (매칭된 승객 수 대비 dropoff한 비율) *1.2)
        matched_passenger_count = len(self.matched_ids)
        cumulative_dropoffs = self.dropped_passengers  # 이미 누적값 포함!
        
        if matched_passenger_count == 0:
            dropoff_ratio = 0
        else:
            dropoff_ratio = cumulative_dropoffs / matched_passenger_count
        
        step_dropoffs = len(all_dropped)
        dropoff_reward = step_dropoffs * self.base_fare * 1.2 * dropoff_ratio
        
        # (c) QOS 비용 = VOT * (대기시간 + 우회시간)
        qos_reward = 0.0
        for p in all_dropped:
           if p.pickup_time is not None and p.dropoff_time == self.time:
               t_wait = max(0, p.pickup_time - p.request_time)
               actual_travel_time = (p.dropoff_time - p.pickup_time)
               t_detour = max(0, actual_travel_time - p.direct_route_time)
               #total_cost += self.alpha_qos * self.VOT * (t_wait + t_detour)
               #total_cost += self.VOT * (t_wait + t_detour)
               qos_reward -= t_wait + t_detour
            
        # (d) 재배치 비용: 이번 스텝에 발생한 재배치 횟수 차이
        rebalancing_in_this_step = self.rebalancing_count - old_rebalancing_count
        rebalancing_reward = - rebalancing_in_this_step
        
        # (e) 배출 가스 비용: 이동 중인 차량 수 × 단가 (여기서는 단가 1로 가정)
        moving_vehicles = sum(1 for v in self.vehicles if v.remaining_travel_time > 0)
        emission_reward = - moving_vehicles

        # 최종 보상 = 비용
        total_cost = matched_reward + dropoff_reward + qos_reward + rebalancing_reward + emission_reward
        reward = total_cost
        
        self.time += 1
        self.last_dropped = all_dropped
        
        return reward

    def generate_passengers_for_current_time(self):
        # 현재 시점에 해당하는 승객만 생성 (self.time이 max_request_time 이하인 경우에만)
        if self.time <= self.max_request_time:
            new_passengers = self.passenger_data[self.passenger_data['Request_time'] == self.time]
            # 중복 생성 방지를 위해 현재 env.passengers에 있는 승객의 ID를 추출
            existing_ids = {p.id for p in self.passengers}
            for _, passenger in new_passengers.iterrows():
                if passenger['User_ID'] not in existing_ids:
                    self.passengers.append(Passenger(
                        id=passenger['User_ID'],
                        start=passenger['Start_node'],
                        end=passenger['End_node'],
                        request_time=passenger['Request_time'],
                        network=self.network  ##[HS] 네트워크 객체 전달
                    ))
                    self.waiting_passenger_count[passenger['Start_node']] += 1  # 대기 승객 수 업데이트

    def get_high_demand_nodes(self):
        recent_requests = [p.start for p in self.passengers if self.time - p.request_time <= 10]
        demand_count = {node: recent_requests.count(node) for node in self.network.graph.nodes}
        historical_data_factor = {node: random.uniform(1.0, 1.5) for node in self.network.graph.nodes}  # 임의의 장기적인 데이터를 반영한 가중치
        for node in demand_count:
            demand_count[node] *= historical_data_factor[node]
        if sum(demand_count.values()) == 0:
            return random.sample(list(self.network.graph.nodes), 3)
        return sorted(demand_count, key=demand_count.get, reverse=True)[:3]

class DQNAgent:
    def __init__(self, state_size, action_size):
        self.state_size = state_size
        self.action_size = action_size
        self.memory = deque(maxlen=10000)  # 메모리 크기 증가
        self.gamma = 0.95
        self.epsilon = 1.0
        self.epsilon_min = 0.05
        self.epsilon_decay = 0.995  # 탐험 감소율 완화
        self.learning_rate = 0.0001  # 학습률 감소
        self.model = self.build_model()
        self.target_model = self.build_model()
        self.update_target_model()
        self.target_update_counter = 0  # 타겟 네트워크 업데이트를 위한 카운터 추가
        
    def build_model(self):
        model = tf.keras.Sequential([
            tf.keras.layers.Input(shape=(self.state_size,)),
            tf.keras.layers.Dense(64, activation='relu'),  # 레이어 유닛 증가
            tf.keras.layers.Dense(64, activation='relu'),
            tf.keras.layers.Dense(self.action_size, activation='linear')
        ])
        model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=self.learning_rate), loss='mse')
        return model
        
    def update_target_model(self):
        self.target_model.set_weights(self.model.get_weights())

    def save_model(self, filepath):
        self.model.save_weights(filepath)
        print(f"Model weights saved at {filepath}")

    def load_model(self, filepath):
        self.model.load_weights(filepath)
        print(f"Model weights loaded from {filepath}")
        
    def act(self, state, valid_nodes, env):
        # 1) epsilon-탐험
        if np.random.rand() <= self.epsilon:
            if valid_nodes and len(valid_nodes) > 0:
                return random.choice(valid_nodes)
            else:
                # valid_nodes가 비었으면 action 없음
                return None
        
        # 2) Q-네트워크 예측
        q_values = self.model.predict(state.reshape(1, -1), verbose=0)[0]
    
        # 3) 고수요 노드, 일반 노드 필터
        high_demand_nodes = []
        normal_nodes = []
        if env is not None and valid_nodes is not None:
            number_of_nodes = len(env.network.graph.nodes)
            number_of_vehicles = len(env.vehicles)
            for node in valid_nodes:
                if (node < number_of_nodes 
                    and (number_of_nodes + number_of_vehicles + node) < len(state)
                    and state[number_of_nodes + number_of_vehicles + node] > 0):
                    high_demand_nodes.append(node)
                else:
                    normal_nodes.append(node)
        else:
            # env가 없거나 valid_nodes가 비었으면 action 없음
            return None
        
        # 4) 고수요 노드 중 최대 Q값과 일반 노드 중 최대 Q값을 비교
        best_q_hd = -1e9
        best_action_hd = None
        if high_demand_nodes:
            for node in high_demand_nodes:
                if node < self.action_size and q_values[node] > best_q_hd:
                    best_q_hd = q_values[node]
                    best_action_hd = node

        best_q_normal = -1e9
        best_action_normal = None
        if normal_nodes:
            for node in normal_nodes:
                if node < self.action_size and q_values[node] > best_q_normal:
                    best_q_normal = q_values[node]
                    best_action_normal = node
        
        if best_q_hd >= best_q_normal:
            if best_action_hd is not None:
                return best_action_hd
            else:
                return best_action_normal if best_action_normal is not None else 0
        else:
            if best_action_normal is not None:
                return best_action_normal
            else:
                return best_action_hd if best_action_hd is not None else 0

    def replay(self, batch_size):
        if len(self.memory) < batch_size:
            return 0
        
        minibatch = random.sample(self.memory, batch_size)
        states = []
        targets = []
        
        for state, action, reward, next_state, done in minibatch:
            # 현재 Q값 (numpy 배열로 변환)
            current_q = self.model(state.reshape(1, -1), training=False)[0].numpy()
            
            if done:
                target_value = reward
            else:
                next_q = self.target_model(next_state.reshape(1, -1), training=False)[0].numpy()
                target_value = reward + self.gamma * np.amax(next_q)
            
            a_idx = min(action, self.action_size - 1)
            current_q[a_idx] = target_value
            
            states.append(state.reshape(-1))
            targets.append(current_q)
        
        states = np.array(states)
        targets = np.array(targets)
        
        # train_on_batch 사용 (tf.function 내부 문제 회피)
        loss = self.model.train_on_batch(states, targets)
        
        self.target_update_counter += 1
        if self.target_update_counter % 50 == 0:
            self.update_target_model()
        
        return loss

    def remember(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))
    
    # 에피소드 종료 시 epsilon 감소
    def decay_epsilon(self):
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay

def main():
    nodes = list(range(1, 25))
    net_file = os.path.join(data_dir, 'SiouxFalls_net.tntp')
    flow_file = os.path.join(data_dir, 'SiouxFalls_flow.tntp')
    node_coord_file = os.path.join(data_dir, 'SiouxFalls_node.tntp')
    node_xy_file = os.path.join(data_dir, 'SiouxFalls_node_xy.tntp')
    network = SiouxFallsNetwork(net_file, flow_file, node_coord_file, node_xy_file)
    
    # 초기화된 travel_time 저장
    travel_time_output = os.path.join(result_dir, 'travel_time.csv')
    network.save_travel_time(travel_time_output)
    
    # OD matrix 생성 및 저장
    od_matrix_output = os.path.join(result_dir, 'od_matrix.csv')
    network.generate_od_matrix(od_matrix_output)
    
    print("Simulation setup complete.")
    
    # 차량 초기 위치 로드
    try:
        vehicle_positions = pd.read_csv(vehicle_positions_file)['initial_position'].tolist()
    except FileNotFoundError:
        print(f"Error: {vehicle_positions_file} not found.")
        return
    except KeyError:
        print(f"Error: 'initial_position' column not found in {vehicle_positions_file}.")
        return

    state_size = len(network.graph.nodes) * 3 + len(vehicle_positions)  # 상태 크기 계산
    valid_nodes = [n for n in network.graph.nodes if n != 0]
    agent = DQNAgent(state_size=state_size, action_size=len(valid_nodes))

    episodes = 350
    batch_size = 32

    results = pd.DataFrame(columns=[
        "Episode", "Step", "Total Reward", "Loss", "Vehicle ID", "Vehicle Location", "Vehicle Status",
        "Passenger ID", "Passenger Start", "Passenger End", "Passenger Request Time", "Passenger Pickup Time",
        "Passenger Dropoff Time"
    ])

    for e in range(episodes):
        # 매 에피소드마다 승객 및 차량 데이터 로드
        passenger_data = pd.read_csv(passenger_data_file)
        
        # RideSharingEnvironment 인스턴스 생성
        env = RideSharingEnvironment(
            network=network,
            num_vehicles=len(vehicle_positions),  # 차량 수는 vehicle_positions의 길이에 기반
            capacity=5,
            passenger_data=passenger_data,
            vehicle_positions=vehicle_positions
        )
        
        state = np.array(env.get_state()).reshape(1, -1)
        total_reward = 0
        env.rebalancing_count = 0
        # matched_passengers와 canceled_passengers는 집합을 통해 관리하므로, 초기화는 __init__에서 처리됨
        env.matched_passengers = 0  
        env.dropped_passengers = 0
        # 에피소드 시작 시, 유니크 ID 집합 초기화
        env.matched_ids = set()
        env.canceled_ids = set()
        episode_data = []
        
        # 동적 스텝: 최대 요청 시간 이후, 모든 차량에 승객이 없을 때까지 진행
        while (env.time <= env.max_request_time) or any(len(vehicle.passengers) > 0 for vehicle in env.vehicles):
            actions = [agent.act(state, valid_nodes, env) for _ in env.vehicles]

            for vehicle in env.vehicles:
                if not vehicle.passengers and not vehicle.current_path:
                    high_demand_nodes = env.get_high_demand_nodes()
                    if high_demand_nodes:
                        closest_node = min(high_demand_nodes, key=lambda node: nx.shortest_path_length(env.network.graph, source=vehicle.current_location, target=node, weight="weight"))
                        vehicle.rebalance_target = closest_node
                        vehicle.rebalance_count += 1
            
            reward = env.step(actions)
            dropped_list = env.last_dropped
            total_reward += reward
            next_state = np.array(env.get_state()).reshape(1, -1)
            done = False  # 동적 종료 조건이므로 여기서는 별도 done 체크 없이 while 조건으로 종료됨
            
            for v_idx, vehicle in enumerate(env.vehicles):
                agent.remember(
                    state,          
                    actions[v_idx],  
                    reward,          # 여기서는 전체 reward를 저장 (원하면 차량별로 변경 가능)
                    next_state,
                    done
                )
            
            loss_value = agent.replay(batch_size)
            
            # 로깅: 차량 내부 승객 정보
            for vehicle in env.vehicles:
                vehicle_status = vehicle.status  # 250321 추가
                for passenger in vehicle.passengers:
                    episode_data.append({
                        "Episode": e + 1,
                        "Step": env.time,
                        "Total Reward": total_reward,
                        "Matched Reward": env.matched_reward,
                        "Dropoff Reward": env.dropoff_reward,
                        "QoS Reward": env.qos_reward,
                        "Rebalancing Reward": env.rebalancing_reward,
                        "Emission Reward": env.emission_reward,
                        "Loss": loss_value,
                        "Vehicle ID": vehicle.id,
                        "Vehicle Location": vehicle.current_location,
                        "Vehicle Status": vehicle_status,
                        "Passenger ID": passenger.id,
                        "Passenger Start": passenger.start,
                        "Passenger End": passenger.end,
                        "Passenger Request Time": passenger.request_time,
                        "Passenger Pickup Time": passenger.pickup_time,
                        "Passenger Dropoff Time": passenger.dropoff_time
                    })

            # 로깅: 하차 승객 정보 (dropped_list 사용)
            for passenger in dropped_list:
                episode_data.append({
                    "Episode": e + 1,
                    "Step": env.time,
                    "Total Reward": total_reward,
                    "Matched Reward": env.matched_reward,
                    "Dropoff Reward": env.dropoff_reward,
                    "QoS Reward": env.qos_reward,
                    "Rebalancing Reward": env.rebalancing_reward,
                    "Emission Reward": env.emission_reward,
                    "Loss": loss_value,
                    "Vehicle ID": "N/A",
                    "Vehicle Location": "N/A",
                    "Vehicle Status": "Dropped Off",
                    "Passenger ID": passenger.id,
                    "Passenger Start": passenger.start,
                    "Passenger End": passenger.end,
                    "Passenger Request Time": passenger.request_time,
                    "Passenger Pickup Time": passenger.pickup_time,
                    "Passenger Dropoff Time": passenger.dropoff_time
                })
            
            state = next_state

        waiting_ids = {p.id for p in env.passengers if p.pickup_time is None}
        total_accounted_passengers = len(env.matched_ids.union(waiting_ids).union(env.canceled_ids))

        print(f"Episode {e + 1}/{episodes},Total Reward: {total_reward}, Loss: {loss_value},"
              #f"Matched Reward: {env.matched_reward}, Dropoff Reward: {env.dropoff_reward}, QoS Reward: {env.qos_reward}, Rebalancing Reward: {env.rebalancing_reward}, Emission Reward: {env.emission_reward},"
              f"Matched Passengers: {len(env.matched_ids)},Dropped Passengers: {env.dropped_passengers},"
              f"Waiting Passengers: {len(waiting_ids)},Canceled Passengers: {len(env.canceled_ids)},"
              f"Total Accounted Passengers: {total_accounted_passengers},Rebalancing Count: {env.rebalancing_count}")

        df_episode = pd.DataFrame(episode_data)
        df_episode.to_csv(os.path.join(result_dir, f"Episode_{e + 1}.csv"), index=False, encoding="utf-8-sig")
        # 에피소드 종료 시 취소 카운터 reset
        env.canceled_passengers = 0

        # 50번마다 모델 저장
        if (e + 1) % 50 == 0:
            model_path = os.path.join(result_dir, f"dqn_model_weights_episode_{e + 1}.weights.h5")
            agent.save_model(model_path)
        
        # 에피소드 종료 시 epsilon 감소
        agent.decay_epsilon()
        
        # 모든 에피소드 종료 후 최종 모델 저장
        if e == episodes - 1:  # 마지막 에피소드
            final_model_path = os.path.join(result_dir, "dqn_model_weights_final.weights.h5")
            agent.save_model(final_model_path)
 

if __name__ == "__main__":
    main()

Travel time data saved to /home/dhshin/kimchanjin/results/travel_time.csv
OD matrix saved to /home/dhshin/kimchanjin/results/od_matrix.csv
Simulation setup complete.


2025-03-26 17:41:07.006437: I tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:606] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.
2025-03-26 17:41:07.601467: I tensorflow/compiler/xla/service/service.cc:168] XLA service 0x1d580810 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-03-26 17:41:07.601512: I tensorflow/compiler/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
2025-03-26 17:41:07.613789: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:255] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-03-26 17:41:07.851829: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:432] Loaded cuDNN version 8907
2025-03-26 17:41:07.986637: I ./tensorflow/compiler/jit/device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Episode 1/350,Total Reward: -3782.305954040993, Loss: 62.049652099609375,Matched Reward: 0.0, Dropoff Reward: 0.0, QoS Reward: 0.0, Rebalancing Reward: 0.0, Emission Reward: 0.0,Matched Passengers: 47,Dropped Passengers: 47,Waiting Passengers: 0,Canceled Passengers: 3,Total Accounted Passengers: 50,Rebalancing Count: 198
Episode 2/350,Total Reward: -3008.1440378399147, Loss: 13.490032196044922,Matched Reward: 0.0, Dropoff Reward: 0.0, QoS Reward: 0.0, Rebalancing Reward: 0.0, Emission Reward: 0.0,Matched Passengers: 47,Dropped Passengers: 47,Waiting Passengers: 0,Canceled Passengers: 3,Total Accounted Passengers: 50,Rebalancing Count: 186
Episode 3/350,Total Reward: -4687.297651511994, Loss: 14.968221664428711,Matched Reward: 0.0, Dropoff Reward: 0.0, QoS Reward: 0.0, Rebalancing Reward: 0.0, Emission Reward: 0.0,Matched Passengers: 50,Dropped Passengers: 50,Waiting Passengers: 0,Canceled Passengers: 0,Total Accounted Passengers: 50,Rebalancing Count: 366
Episode 4/350,Total Reward: -4


KeyboardInterrupt

